# How WALRUS samples training windows, and how often they repeat

Companion to `mydocs/walrus_sampling_and_repetition.md`. Every claim tagged **[verified]** there
is checked by an `assert` here, and the section numbers (E1–E8) match the doc.

Most experiments use the **real** `BatchedMultisetSampler`, the **real** `torch` `RandomSampler`
and `DataLoader`, and a stand-in dataset that only returns the indices it was asked for. That is
enough, because everything that decides *which* window is drawn happens in the sampler and the
loader, not in the HDF5 reading. Section E7 then checks the index → (file, trajectory, start
frame, `dt`) mapping on a real dataset, and E8 turns all of it into repetition estimates for the
PT9 runs.

Runs on the login node on CPU. Only E7 reads data.

In [1]:
import itertools
import math
import sys
from collections import Counter, deque
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, DistributedSampler, RandomSampler

REPO = Path.cwd().parent if Path.cwd().name == "demo_notebooks" else Path.cwd()
sys.path[:0] = [str(REPO / "scripts"), str(REPO / "LeonardoRunScripts" / "PT9_2d")]

from walrus.data.mixed_dset_sampler import BatchedMultisetSampler
from walrus.utils.experiment_utils import seed_all

print("torch", torch.__version__)  # the RNG details below are specific to this version's sampler/dataloader code

/scratch_local/ipykernel_3520968/3044105240.py:8: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


torch 2.5.1+cu121


## E0. A stand-in for `MixedWellDataset`

`BatchedMultisetSampler` only reads three things from the dataset: `sub_dsets` (to build one
`RandomSampler` per dataset, which only needs `len`), `offsets` (to turn a local index into a
global one) and `effective_batch_sizes`. `MixedWellDataset.__getitem__` turns the global indices
back into `(dataset, local indices)` with `np.searchsorted(offsets, indices[0])`
(`walrus/data/multidataset.py:374-385`). The stand-in does exactly that and returns the result
instead of loading data.

`make_loader` builds the loader exactly as `MixedWellDataModule.train_dataloader` does
(`walrus/data/multidatamodule.py:410-438`): `batch_size=None`, `collate_fn=None`,
`drop_last=False`, `recycle=True`, `distributed=True`. `run` mirrors the training loop:
`set_epoch`, then one `next()` per micro-batch. After each micro-batch it makes `jitter` calls
to `torch.randint` on the global RNG, standing in for the forward pass. The PT9 jitterer
(`FixedPatchJittererBoundaryPad`, `patch_jitterers.py:460`) makes one such call per non-singleton
axis, so 2 per forward on 2D data.

In [2]:
class FakeSub:
    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n


class FakeMixed(Dataset):
    def __init__(self, sizes):
        self.sub_dsets = [FakeSub(n) for n in sizes]
        self.offsets = [0]
        for n in sizes:
            self.offsets.append(self.offsets[-1] + n)
        self.offsets[0] = -1  # as in multidataset.py:302
        self.effective_batch_sizes = [1.0] * len(sizes)

    def __len__(self):
        return self.offsets[-1]

    def __getitem__(self, indices):
        d = int(np.searchsorted(self.offsets, indices[0], side="right") - 1)
        return {"dataset": d, "local": [i - max(self.offsets[d], 0) for i in indices]}


def make_loader(sizes, B, max_samples, sampling_rank=0, workers=0, base_sampler=RandomSampler):
    sampler = BatchedMultisetSampler(FakeMixed(sizes), base_sampler, B, distributed=True,
                                     max_samples=max_samples, recycle=True, drop_last=False,
                                     rank=sampling_rank)
    return DataLoader(sampler.dataset, num_workers=workers, batch_size=None, sampler=sampler,
                      collate_fn=None)


def run(sizes, B, max_samples, epochs, rank_seed, sampling_rank=0, workers=0, jitter=2,
        base_sampler=RandomSampler):
    # One rank: seed as train.py:155, then train epoch by epoch. Returns {epoch: [(dataset, [local idx])]}.
    seed_all(rank_seed)
    loader = make_loader(sizes, B, max_samples, sampling_rank, workers, base_sampler)
    log = {}
    for e in epochs:
        loader.sampler.set_epoch(e)
        steps = []
        for batch in loader:
            steps.append((batch["dataset"], batch["local"]))
            for _ in range(jitter):
                torch.randint(-3, 4, ())  # forward-pass jitter draws on the global CPU RNG
        log[e] = steps
    return log


def rank_seed(seed, rank):
    return seed + 1_000_003 * rank  # train.py:155

## E1. An exact model of every random draw

Before checking properties one by one, here is the whole mechanism written out as a predictor.
It uses **no** walrus or `DataLoader` code, only the same global-RNG calls in the same order. If
it reproduces the real loader draw for draw, then each step below is exactly what happens:

1. **Iterator start** (`iter(loader)`, once per epoch). The DataLoader draws its worker
   `_base_seed` from the global RNG (`torch/utils/data/dataloader.py:666`).
2. **Dataset choice.** The sampler's *own* generator is seeded `100·epoch + 10·0 + sampling_rank`
   (`mixed_dset_sampler.py:172`) and picks uniformly over the datasets each micro-batch.
3. **Window choice.** Each dataset's `RandomSampler` is a generator. On its *first* `next()`
   after being (re)created, it draws a seed from the global RNG and makes a `randperm` of the
   dataset's windows (`torch/utils/data/sampler.py:175-199`). It is recreated at the start of
   every epoch (`mixed_dset_sampler.py:168`) and when it runs out (recycle, `:195-206`).
4. **Recycle.** If a dataset runs out part-way through a batch, the partial batch is thrown
   away, the counter is decremented, and the loop draws a **new dataset**.
5. **Loader ordering.** With `workers = 0`, the next batch's indices are drawn just before it
   is returned. With `workers > 0`, `2·workers` batches are drawn up front (`dataloader.py:1227`),
   then one more each time a batch is returned (`_process_data`, `:1487-1489`). In both cases
   that is *before* the forward pass of the batch being returned.

In [3]:
def rand_global():
    return int(torch.empty((), dtype=torch.int64).random_().item())  # how RandomSampler / DataLoader seed


def predict(sizes, B, max_samples, epochs, rank_seed, sampling_rank=0, workers=0, jitter=2):
    seed_all(rank_seed)
    K, log = len(sizes), {}
    for e in epochs:
        rand_global()                                    # 1. DataLoader _base_seed
        g = torch.Generator()
        g.manual_seed(100 * e + sampling_rank)           # 2. dataset-choice generator
        perms = [None] * K                               # 3. lazily seeded permutations

        def next_idx(d):
            if perms[d] is None:
                gen = torch.Generator()
                gen.manual_seed(rand_global())
                perms[d] = [torch.randperm(sizes[d], generator=gen).tolist(), 0]
            perm, p = perms[d]
            if p >= len(perm):
                raise StopIteration
            perms[d][1] += 1
            return perm[p]

        def sampler():
            count = 0
            while True:
                count += 1
                d = int(torch.randint(0, K, (1,), generator=g).item())
                try:
                    batch = [next_idx(d) for _ in range(B)]
                except StopIteration:                    # 4. recycle: drop partial batch, redraw dataset
                    count -= 1
                    perms[d] = None
                    continue
                yield d, batch
                if count >= max_samples:
                    return

        gen, steps = sampler(), []
        queue = deque(itertools.islice(gen, 2 * workers))  # 5. prefetch (empty when workers=0)
        while True:
            if workers == 0:
                item = next(gen, None)
            else:
                item = queue.popleft() if queue else None
                nxt = next(gen, None)
                if nxt is not None:
                    queue.append(nxt)
            if item is None:
                break
            steps.append(item)
            for _ in range(jitter):
                torch.randint(-3, 4, ())
        log[e] = steps
    return log


# Small datasets on purpose, so recycling happens within an epoch (dataset 2 has only 25 windows).
SIZES, B, MS = [5000, 300, 25], 2, 400
for workers, jitter in [(0, 0), (0, 2), (2, 0), (2, 2), (4, 2)]:
    for rank in range(3):
        real = run(SIZES, B, MS, range(1, 4), rank_seed(0, rank), workers=workers, jitter=jitter)
        pred = predict(SIZES, B, MS, range(1, 4), rank_seed(0, rank), workers=workers, jitter=jitter)
        assert real == pred, (workers, jitter, rank)
print("E1: predictor == real loader, draw for draw (3 epochs, 3 ranks, with recycling, workers 0/2/4, jitter 0/2)")

E1: predictor == real loader, draw for draw (3 epochs, 3 ranks, with recycling, workers 0/2/4, jitter 0/2)


The predictor matches exactly. Two consequences follow directly from the code above:

- The window drawn at any step depends on **how many global-RNG draws happened before it**,
  including every forward pass (`jitter`) of every earlier step and epoch. With `workers = 0`
  that holds even inside epoch 1. With `workers > 0`, the datasets that first appear in the
  prefetch window are seeded before any forward pass.
- Exact *offline replay* of a real run is possible only if you replicate every global CPU RNG
  draw the trainer makes: forward-pass jitter (in training and validation), the validation
  loaders' `_base_seed` draws, and anything else. That is fragile, which is why the doc
  recommends logging draws during the run instead.

## E2. Which dataset: the same on every GPU and for every seed

Under `fsdp` the sampling rank is 0 on every GPU (`training.py:369-376`), and the sampler's seed
argument is never passed (`multidatamodule.py:419`). So the dataset sequence depends only on the
epoch. It also matches `count_WALRUS_flops.replay_epoch`, the offline replay notebook 3 uses.

In [4]:
import count_WALRUS_flops as cwf

SIZES9 = [9000] * 9  # big enough that nothing recycles within an epoch
seqs = {}
for seed, rank in [(0, 0), (0, 1), (0, 3), (1, 0), (7, 2)]:
    log = run(SIZES9, 2, 500, [1, 2], rank_seed(seed, rank), workers=2)
    seqs[(seed, rank)] = {e: [d for d, _ in log[e]] for e in log}
first = next(iter(seqs.values()))
assert all(s == first for s in seqs.values())
assert first[1] != first[2]  # but it changes every epoch
for e in (1, 2):
    counts = Counter(first[e])
    assert [counts[d] for d in range(9)] == cwf.replay_epoch(100 * e, [2] * 9, SIZES9, 500, True)
print("E2: dataset sequence identical across ranks and seeds, different across epochs, equal to replay_epoch")

E2: dataset sequence identical across ranks and seeds, different across epochs, equal to replay_epoch


## E3. Within a pass, across epochs, and on recycle

- **Within one pass** of a dataset's permutation, no window repeats.
- **Each epoch** starts a new random permutation. It is not a continuation of the previous one,
  so an epoch that draws only a small part of a dataset does *not* make the next epoch draw a
  different part.
- **On recycle**, a new permutation starts and the partial batch is dropped. A window can
  therefore be drawn twice within one epoch only if the dataset ran out in that epoch.

In [5]:
log = run([5000, 300, 25], 2, 400, [1, 2], rank_seed(0, 0), workers=2)


def draws(steps, d):
    return [i for dd, b in steps if dd == d for i in b]


# Dataset 0 (5000 windows) never runs out: no window repeats within an epoch.
big1, big2 = draws(log[1], 0), draws(log[2], 0)
assert len(big1) == len(set(big1)) and len(big2) == len(set(big2))
overlap = len(set(big1) & set(big2))
print(f"dataset 0 (5000 windows): {len(big1)} draws in epoch 1, {len(big2)} in epoch 2, "
      f"{overlap} windows in common (independent subsets expect ≈ {len(big1) * len(big2) / 5000:.0f})")

# Dataset 2 (25 windows) runs out many times per epoch. Each pass yields 12 batches = 24 windows
# (the 13th batch fails after taking the 25th window, which is dropped), all distinct, then a new
# permutation starts.
small = draws(log[1], 2)
blocks = [small[i:i + 24] for i in range(0, len(small), 24)]
full, last = blocks[:-1] if len(blocks[-1]) < 24 else blocks, blocks[-1]  # the epoch can end mid-pass
assert all(len(set(b)) == 24 for b in full) and len(set(last)) == len(last)
assert len({tuple(b) for b in full}) == len(full)  # every pass is a different permutation
assert all(len(set(range(25)) - set(b)) == 1 for b in full)
print(f"dataset 2 (25 windows): {len(full)} complete passes in epoch 1 (+ {len(last) if len(last) < 24 else 0} "
      f"draws of an unfinished one), 24 distinct windows each, one window per pass never yielded, "
      f"every pass a new order")

dataset 0 (5000 windows): 312 draws in epoch 1, 284 in epoch 2, 21 windows in common (independent subsets expect ≈ 18)
dataset 2 (25 windows): 9 complete passes in epoch 1 (+ 14 draws of an unfinished one), 24 distinct windows each, one window per pass never yielded, every pass a new order


## E4. Forward passes change which windows later epochs draw

Same seed and same rank, with 0 vs. 2 jitter draws per micro-batch. The datasets seeded in the
prefetch window of epoch 1 are unaffected. Everything after that depends on the forward passes.

In [6]:
SIZES3 = [5000, 4000, 3000]
a = run(SIZES3, 2, 300, [1, 2], rank_seed(0, 0), workers=4, jitter=0)
b = run(SIZES3, 2, 300, [1, 2], rank_seed(0, 0), workers=4, jitter=2)
g = torch.Generator(); g.manual_seed(100)
seq1 = [int(torch.randint(0, 3, (1,), generator=g).item()) for _ in range(8)]
all_in_prefetch = len(set(seq1)) == 3  # do all 3 datasets appear among the 8 batches prefetched in epoch 1?
assert (a[1] == b[1]) == all_in_prefetch
assert a[2] != b[2]
c = run(SIZES3, 2, 300, [1], rank_seed(0, 0), workers=0, jitter=0)
d = run(SIZES3, 2, 300, [1], rank_seed(0, 0), workers=0, jitter=2)
print(f"workers=4: all datasets in the prefetch window: {all_in_prefetch}; epoch 1 identical: {a[1] == b[1]}; "
      f"epoch 2 identical: {a[2] == b[2]}")
print(f"workers=0: epoch 1 identical: {c[1] == d[1]} (each dataset is seeded at its first draw, after earlier forwards)")

workers=4: all datasets in the prefetch window: True; epoch 1 identical: True; epoch 2 identical: False
workers=0: epoch 1 identical: False (each dataset is seeded at its first draw, after earlier forwards)


## E5. Across GPUs (and nodes): independent draws, nothing removes duplicates

Every rank seeds its global RNG with `seed + 1_000_003·rank`, so each rank's permutations are
independent. There is no coordination between ranks. Nodes are just more ranks, so the same
holds across nodes. If each of two ranks draws a random `r`-subset of `W` windows in an epoch,
the expected overlap is `r²/W` (hypergeometric mean). Under fsdp the ranks also draw from the
**same dataset at the same step**, so the expected number of rank pairs sharing a window *in the
same micro-batch step* is `C(R,2)·B²/W` per step.

In [7]:
SIZES2, B, MS, R = [3000, 50000], 2, 2000, 4
logs = [run(SIZES2, B, MS, [1], rank_seed(0, r), workers=2)[1] for r in range(R)]
rows = []
for d, W in enumerate(SIZES2):
    sets = [{i for dd, bb in lg if dd == d for i in bb} for lg in logs]
    r = np.mean([len(s) for s in sets])
    pair = np.mean([len(x & y) for x, y in itertools.combinations(sets, 2)])
    same_step = sum(
        len(set(logs[i][t][1]) & set(logs[j][t][1]))
        for t in range(MS) if logs[0][t][0] == d
        for i, j in itertools.combinations(range(R), 2)
    )
    n_steps_d = sum(1 for t in range(MS) if logs[0][t][0] == d)
    rows.append({"W": W, "draws/rank r": r, "pair overlap": pair, "expected r²/W": r * r / W,
                 "same-step shared": same_step, "expected C(R,2)·B²/W·steps": math.comb(R, 2) * B * B / W * n_steps_d})
df = pd.DataFrame(rows)
assert all(abs(df["pair overlap"] - df["expected r²/W"]) < 4 * np.sqrt(df["expected r²/W"]) + 1)
df.round(1)

,W,draws/rank r,pair overlap,expected r²/W,same-step shared,"expected C(R,2)·B²/W·steps"
0,3000,2038.0,1391.5,1384.5,12,8.2
1,50000,1962.0,77.8,77.0,0,0.5


## E6. The alternative: `allow_sharding_in_train=True` (`DistributedSampler`)

Off by default and not used in PT9. With it, each dataset is split across ranks: one shared
permutation seeded `0 + epoch` (not `cfg.seed`), with rank `r` taking every `R`-th element
(`torch/utils/data/distributed.py:106-133`). Ranks are then **disjoint** within a pass. Because
the permutation depends only on the epoch, a recycle within an epoch **replays the same order**.

In [8]:
def ds_factory(R, r):
    class RankDS(DistributedSampler):
        def __init__(self, dataset, drop_last=False):
            super().__init__(dataset, num_replicas=R, rank=r, drop_last=drop_last)
    return RankDS


R, SIZES2 = 4, [4000, 30]
logs = [run(SIZES2, 2, 200, [1], rank_seed(0, r), workers=0, base_sampler=ds_factory(R, r))[1] for r in range(R)]
big = [{i for d, b in lg if d == 0 for i in b} for lg in logs]
assert all(not (x & y) for x, y in itertools.combinations(big, 2))           # disjoint across ranks
other_seed = run(SIZES2, 2, 200, [1], rank_seed(5, 0), workers=0, base_sampler=ds_factory(R, 0))[1]
assert other_seed == logs[0]                                                 # independent of cfg.seed
small = [i for d, b in logs[0] if d == 1 for i in b]
per_pass = math.ceil(30 / R) // 2 * 2
assert small[:per_pass] == small[per_pass:2 * per_pass]                      # recycle replays the order
print(f"disjoint across ranks; same for every seed; rank 0's share of the 30-window dataset: {small[:per_pass]} … repeated")

disjoint across ranks; same for every seed; rank 0's share of the 30-window dataset: [25, 23, 19, 22, 9, 10, 21, 16] … repeated


## E6b. Resuming a job replays the RNG from the seed

Checkpoints store no RNG state (`walrus/trainer/checkpoints.py`), and `train.py:147-155` re-runs
`seed_all` at start-up. A resumed job's global RNG therefore starts in the same state as job 1's.
At its first epoch `k+1`, the first `2·workers` prefetched batches draw the same sequence of
`RandomSampler` seeds as job 1's epoch 1, handed out to datasets **in order of first
appearance**. Each dataset's seed is the global-RNG draw it reaches first. A dataset that reaches its seed
draw after the **same sequence of earlier calls** in both epochs gets the **same permutation**, so
its draws in epoch `k+1` are exactly the windows it drew in epoch 1, in the same order. In
practice that means first appearing at the same position within the prefetch window, before any
forward pass has run.

Demo with the 9 PT9 datasets at their real window counts and `workers = 10` (what most PT9 runs
use): job 1 runs epochs 1–3, then job 2 resumes at epoch 4.

In [9]:
import well_registry as wr
from pt9_substrate import PRETRAIN_DATASETS

reg = wr.load_registry(REPO / "demo_notebooks" / "data" / "dataset_table_2d.json")["datasets"]
W9 = [wr.n_windows(reg[n]["splits"]["train"], 6, 1, 1) for n in PRETRAIN_DATASETS]
B, MS, WORKERS = 2, 2000, 10


def seed_prefixes(seq, workers=WORKERS, jitter=2):
    # For one epoch's dataset sequence, the sequence of global-RNG calls made before each dataset's
    # RandomSampler seed draw ('b' base seed, 's' a seed, 'j' a forward-pass draw). Two epochs that
    # start from the same RNG state give a dataset the same seed iff these prefixes are equal.
    calls, prefix = ["b"], {}

    def put(t):
        if t < len(seq) and seq[t] not in prefix:
            prefix[seq[t]] = "".join(calls)
            calls.append("s")

    for t in range(2 * workers):
        put(t)
    for t in range(len(seq)):
        put(t + 2 * workers)
        calls.extend("j" * jitter)
    return prefix


def dataset_seq(e, K=9, n=MS):
    g = torch.Generator()
    g.manual_seed(100 * e)
    return [int(torch.randint(0, K, (1,), generator=g).item()) for _ in range(n)]


def identical_after_resume(e_resume, rank=0):
    job1 = run(W9, B, MS, [1], rank_seed(0, rank), workers=WORKERS)[1]
    job2 = run(W9, B, MS, [e_resume], rank_seed(0, rank), workers=WORKERS)[e_resume]  # fresh seed_all = resume
    p1, p2 = seed_prefixes([d for d, _ in job1]), seed_prefixes([d for d, _ in job2])
    rows = []
    for d, n in enumerate(PRETRAIN_DATASETS):
        a = [i for dd, b in job1 if dd == d for i in b]
        b_ = [i for dd, b in job2 if dd == d for i in b]
        m = min(len(a), len(b_))
        same = sum(x == y for x, y in zip(a[:m], b_[:m]))
        predicted = p1.get(d) is not None and p1.get(d) == p2.get(d)
        rows.append({"dataset": n, "seed drawn after (epoch 1)": len(p1.get(d, "")),
                     f"… (epoch {e_resume})": len(p2.get(d, "")),
                     "predicted identical": predicted, "same draw, same position": same, "of": m})
        assert (same == m) if predicted else (same < 0.02 * m + 5)
    return pd.DataFrame(rows).set_index("dataset")


identical_after_resume(4)

,seed drawn after (epoch 1),… (epoch 4),predicted identical,"same draw, same position",of
dataset,,,,,
acoustic_scattering_discontinuous,7,4,False,0,470
acoustic_scattering_inclusions,3,5,False,0,416
acoustic_scattering_maze,5,1,False,0,432
euler_multi_quadrants_openBC,8,7,False,0,476
euler_multi_quadrants_periodicBC,61,8,False,0,364
gray_scott_reaction_diffusion,6,3,False,0,404
planetswe,4,2,False,0,440
rayleigh_benard,1,33,False,0,422
shear_flow,2,6,False,0,418


Which datasets repeat depends only on the two epochs' dataset sequences (and `workers`, and the
2 jitter draws per forward), and the dataset sequences are deterministic (E2). So the effect of a
resume at any epoch can be computed offline without running anything. For each possible resume
epoch, count the datasets that would replay their epoch-1 permutation:

In [10]:
p1 = seed_prefixes(dataset_seq(1))
hits = np.array([sum(1 for d in range(9) if d in p1 and p1[d] == seed_prefixes(dataset_seq(e)).get(d))
                 for e in range(2, 101)])
print(f"resume at epoch 2..100: identical-permutation datasets per resume: mean {hits.mean():.2f}, "
      f"max {hits.max()}, P(at least one) = {(hits > 0).mean():.0%}")

resume at epoch 2..100: identical-permutation datasets per resume: mean 0.90, max 4, P(at least one) = 61%


The epoch-4 resume above happens to replay no dataset. Here is the first resume epoch where the offline count predicts at least one, run for real. The predicted datasets must replay every one of their epoch-1 draws in the same order:

In [11]:
e_hit = 2 + int(np.argmax(hits > 0))
demo = identical_after_resume(e_hit)
assert demo["predicted identical"].sum() == hits[e_hit - 2] > 0
print(f"resume at epoch {e_hit}")
demo

resume at epoch 7


,seed drawn after (epoch 1),… (epoch 7),predicted identical,"same draw, same position",of
dataset,,,,,
acoustic_scattering_discontinuous,7,7,True,404,404
acoustic_scattering_inclusions,3,4,False,0,426
acoustic_scattering_maze,5,6,False,0,430
euler_multi_quadrants_openBC,8,1,False,0,438
euler_multi_quadrants_periodicBC,61,2,False,0,364
gray_scott_reaction_diffusion,6,3,False,0,414
planetswe,4,9,False,0,440
rayleigh_benard,1,5,False,0,422
shear_flow,2,8,False,0,418


## E7. From a global index to (file, trajectory, start frame, `dt`) on real data

`the_well/data/datasets.py:780-813`, `_load_one_sample(index)`:

```
file_idx   = searchsorted(file_index_offsets, index, side="right") - 1
local      = index - file_index_offsets[file_idx]          # offsets[0] is -1, clamped to 0
traj       = local // windows_per_trajectory[file_idx]       # = n_steps - (T_in + T_out) + 1
t0         = local %  windows_per_trajectory[file_idx]
dt         = randint(1, min(max_dt, (n_steps - t0 - 1) // (T_in + T_out - 1)) + 1)   # in the worker, np.random
```

Checked on VE, the one dataset whose files have different trajectory lengths (20 and 60 steps).
The hand mapping uses only the per-file list from the registry.

In [12]:
from walrus.data.inflated_dataset import InflatedWellDataset

ROOT = Path(wr.load_registry(REPO / "demo_notebooks" / "data" / "dataset_table_2d.json")["root"])
ve = InflatedWellDataset(path=str(ROOT / "viscoelastic_instability"), well_split_name="train",
                         n_steps_input=6, n_steps_output=1, min_dt_stride=1, max_dt_stride=5,
                         pad_cartesian_data_to_d=3)
files = reg["viscoelastic_instability"]["splits"]["train"]["files"]
assert [Path(p).name for p in ve.files_paths] == [f["name"] for f in files]


def by_hand(index):
    for f_idx, f in enumerate(files):
        w = f["n_steps"] - 7 + 1
        if index < f["n_traj"] * w:
            return f_idx, index // w, index % w
        index -= f["n_traj"] * w


assert len(ve) == sum(f["n_traj"] * (f["n_steps"] - 6) for f in files)
rng = np.random.default_rng(0)
probe = sorted(set(rng.integers(0, len(ve), 12).tolist()) | {0, len(ve) - 1})
rows = []
for i in probe:
    _, f_idx, traj, t0, dt = ve._load_one_sample(i)
    assert (f_idx, traj, t0) == by_hand(i)
    rows.append({"index": i, "file": files[f_idx]["name"], "n_steps": files[f_idx]["n_steps"],
                 "traj": traj, "t0": t0, "dt": dt})
pd.DataFrame(rows)

,index,file,n_steps,traj,t0,dt
0,0,viscoelastic_instability_AH.hdf5,20,0,0,1
1,89,viscoelastic_instability_AH.hdf5,20,6,5,2
2,222,viscoelastic_instability_AH.hdf5,20,15,12,1
3,407,viscoelastic_instability_CAR.hdf5,60,3,21,1
4,950,viscoelastic_instability_CAR.hdf5,60,13,24,4
5,1462,viscoelastic_instability_CAR.hdf5,60,22,50,1
6,1669,viscoelastic_instability_CAR.hdf5,60,26,41,2
7,2771,viscoelastic_instability_EIT.hdf5,60,14,9,4
8,3453,viscoelastic_instability_EIT.hdf5,60,26,43,2
9,3521,viscoelastic_instability_transtochaos_EIT_L.hdf5,20,0,3,1


In [13]:
from the_well.data.utils import maximum_stride_for_initial_index

# dt depends on the start frame: early windows can stretch up to dt=5, late ones are forced to dt=1.
f60 = next(i for i, f in enumerate(files) if f["n_steps"] == 60)
base_index = sum(f["n_traj"] * (f["n_steps"] - 6) for f in files[:f60])
rows = []
for t0 in (0, 20, 35, 45, 53):
    cap = min(5, maximum_stride_for_initial_index(t0, 60, 6, 1))
    dts = Counter(ve._load_one_sample(base_index + t0)[4] for _ in range(12))
    assert set(dts) <= set(range(1, cap + 1))
    rows.append({"t0": t0, "allowed dt": f"1..{cap}", "observed (12 loads)": dict(sorted(dts.items()))})
pd.DataFrame(rows)

,t0,allowed dt,observed (12 loads)
0,0,1..5,"{1: 4, 2: 4, 3: 1, 4: 1, 5: 2}"
1,20,1..5,"{1: 4, 2: 2, 3: 1, 4: 5}"
2,35,1..4,"{1: 2, 2: 2, 3: 3, 4: 5}"
3,45,1..2,"{1: 8, 2: 4}"
4,53,1..1,{1: 12}


## E8. Estimating repetition for the PT9 runs

From E1–E6: under fsdp each rank, each epoch and each dataset draws an independent uniform
random subset of `r_{d,e} = B·c_{d,e}` windows (no repeats inside it, assuming no recycle), where
`c_{d,e}` is the number of micro-batches of dataset `d` in epoch `e`, the same on every rank (E2).
Over `R` ranks and `E` epochs, the number of times a given window is drawn is a sum of
`R·E` independent Bernoulli(`r/W`) indicators, so approximately Poisson with mean
`λ = N/W` (`N` = total draws). That gives:

| quantity | formula |
|---|---|
| draws `N_d` | `R · Σ_e B·c_{d,e}` |
| distinct windows drawn | `W·[1 − Π_e (1 − r_{d,e}/W)^R] ≈ W(1 − e^{−λ})` |
| exact-repeat draws | `N − distinct` (draws of a window already drawn before) |
| share of windows drawn ≥ 2 times | `≈ 1 − e^{−λ}(1 + λ)` |
| same-step shared windows per epoch (fsdp) | `c_{d,e} · C(R,2) · B² / W` |
| pairwise rank overlap per epoch | `r²/W` |

**Frame overlap.** Two different windows of the same trajectory share input frames when their
start frames are close (with `dt = 1`, `|t0 − t0'| ≤ 5`). That is not countable in closed form
once `dt` varies, so it is simulated: draws in the same order as the real sampler (rank-epoch
blocks), each with a `dt` drawn by the real rule, counting a draw as "frame-overlapping" if any of
its 6 input frames was an input frame of an earlier draw.

In [14]:
CODE = {"acoustic_scattering_discontinuous": "AD", "acoustic_scattering_inclusions": "AI",
        "acoustic_scattering_maze": "AM", "euler_multi_quadrants_openBC": "EMQ-O",
        "euler_multi_quadrants_periodicBC": "EMQ-P", "gray_scott_reaction_diffusion": "GS",
        "planetswe": "PS", "rayleigh_benard": "RB", "shear_flow": "SF", "active_matter": "ACT",
        "helmholtz_staircase": "HS", "turbulent_radiative_layer_2D": "TRL2", "viscoelastic_instability": "VE"}
B, T_IN, T_OUT, MAX_DT, MS, E = 2, 6, 1, 5, 2000, 100


def window_table(name):
    # Map window index -> (trajectory id, t0, n_steps) for a whole train split.
    traj, t0, nst, base = [], [], [], 0
    for g in reg[name]["splits"]["train"]["steps"]:
        w = g["n_steps"] - (T_IN + T_OUT) + 1
        idx = np.arange(g["n_traj"] * w)
        traj.append(base + idx // w); t0.append(idx % w); nst.append(np.full(idx.size, g["n_steps"]))
        base += g["n_traj"]
    return np.concatenate(traj), np.concatenate(t0), np.concatenate(nst), base


def simulate(name, per_epoch_draws, R, seed=0, block=8):
    # Exact-repeat and frame-overlap fractions over a run. per_epoch_draws[e] = windows drawn per rank in epoch e.
    rng = np.random.default_rng(seed)
    traj, t0, nst, n_traj = window_table(name)
    W = traj.size
    seen_w = np.zeros(W, bool)
    seen_f = np.zeros((n_traj, int(nst.max())), bool)
    rep = ovl = total = 0
    for r_e in per_epoch_draws:
        for _ in range(R):
            idx = rng.choice(W, size=r_e, replace=False) if r_e <= W else rng.integers(0, W, r_e)
            cap = np.minimum((nst[idx] - t0[idx] - 1) // (T_IN + T_OUT - 1), MAX_DT)
            dt = np.where(cap > 1, rng.integers(1, np.maximum(cap, 1) + 1), 1)
            frames = t0[idx, None] + dt[:, None] * np.arange(T_IN)
            tr = traj[idx]
            for s in range(0, r_e, block):  # small blocks so overlaps between nearby draws are counted
                sl = slice(s, s + block)
                rep += seen_w[idx[sl]].sum()
                ovl += seen_f[tr[sl, None], frames[sl]].any(axis=1).sum()
                seen_w[idx[sl]] = True
                seen_f[tr[sl, None], frames[sl]] = True
            total += r_e
    return rep / total, ovl / total, seen_w.mean(), seen_f[:, :].sum() / sum(
        g["n_traj"] * g["n_steps"] for g in reg[name]["splits"]["train"]["steps"])


def estimates(names, R, epochs=E, K=None):
    K = K or len(names)
    windows = [wr.n_windows(reg[n]["splits"]["train"], T_IN, T_OUT, 1) for n in names]
    per_epoch = np.array([cwf.replay_epoch(100 * e, [B] * K, windows, MS, True) for e in range(1, epochs + 1)])  # (E, K)
    rows = []
    for d, n in enumerate(names):
        W, c = windows[d], per_epoch[:, d]
        r = B * c
        N = R * r.sum()
        lam = N / W
        distinct = W * (1 - np.prod((1 - np.minimum(r, W) / W) ** R))
        rep_sim, ovl_sim, cov_sim, frame_cov = simulate(n, r, R)
        rows.append({
            "dataset": CODE[n], "W": W, "draws/rank/epoch": r.mean(), "λ = N/W": lam,
            "window cov.": distinct / W, "cov. (sim)": cov_sim,
            "exact-repeat draws": 1 - distinct / N, "repeat (sim)": rep_sim,
            "windows drawn ≥2×": 1 - math.exp(-lam) * (1 + lam),
            "frame-overlap draws (sim)": ovl_sim,
            "same-step shared / epoch": c.mean() * math.comb(R, 2) * B * B / W,
            "pair overlap / epoch": r.mean() ** 2 / W,
        })
        assert abs(rows[-1]["window cov."] - cov_sim) < 0.01 and abs(rows[-1]["exact-repeat draws"] - rep_sim) < 0.01
    return pd.DataFrame(rows).set_index("dataset")


FMT = {"W": "{:,}", "draws/rank/epoch": "{:.0f}", "λ = N/W": "{:.2f}", "window cov.": "{:.1%}",
       "cov. (sim)": "{:.1%}", "exact-repeat draws": "{:.1%}", "repeat (sim)": "{:.1%}",
       "windows drawn ≥2×": "{:.1%}", "frame-overlap draws (sim)": "{:.1%}",
       "same-step shared / epoch": "{:.3f}", "pair overlap / epoch": "{:.2f}"}
est_n1 = estimates(PRETRAIN_DATASETS, R=4)
est_n1.style.format(FMT).set_caption("PT9 N1: 4 GPUs, fsdp, 9 datasets, 100 epochs")

,W,draws/rank/epoch,λ = N/W,window cov.,cov. (sim),exact-repeat draws,repeat (sim),windows drawn ≥2×,frame-overlap draws (sim),same-step shared / epoch,pair overlap / epoch
dataset,,,,,,,,,,,
AD,"153,600",450,1.17,69.0%,69.0%,41.0%,41.1%,32.7%,95.8%,0.035,1.32
AI,"268,608",447,0.67,48.6%,48.6%,26.9%,27.0%,14.4%,92.6%,0.020,0.74
AM,"313,600",441,0.56,43.0%,43.0%,23.5%,23.5%,11.0%,91.7%,0.017,0.62
EMQ-O,"380,000",450,0.47,37.8%,37.8%,20.3%,20.2%,8.2%,89.6%,0.014,0.53
EMQ-P,"380,000",442,0.47,37.2%,37.3%,20.0%,19.9%,8.0%,89.4%,0.014,0.52
GS,"955,200",442,0.19,16.9%,16.9%,8.7%,8.8%,1.5%,76.6%,0.006,0.20
PS,"96,192",441,1.83,84.1%,84.0%,54.1%,54.2%,54.7%,97.6%,0.055,2.02
RB,"271,600",443,0.65,48.0%,48.0%,26.5%,26.5%,14.0%,92.9%,0.020,0.72
SF,"173,824",444,1.02,64.0%,63.9%,37.3%,37.4%,27.2%,95.5%,0.031,1.13


In [15]:
est_n2 = estimates(PRETRAIN_DATASETS, R=8)
est_n2.style.format(FMT).set_caption("PT9 N2: 8 GPUs (2 nodes), fsdp, 9 datasets, 100 epochs")

,W,draws/rank/epoch,λ = N/W,window cov.,cov. (sim),exact-repeat draws,repeat (sim),windows drawn ≥2×,frame-overlap draws (sim),same-step shared / epoch,pair overlap / epoch
dataset,,,,,,,,,,,
AD,"153,600",450,2.34,90.4%,90.3%,61.4%,61.4%,67.9%,97.9%,0.164,1.32
AI,"268,608",447,1.33,73.6%,73.6%,44.7%,44.7%,38.4%,96.3%,0.093,0.74
AM,"313,600",441,1.12,67.5%,67.6%,39.9%,39.8%,31.0%,95.9%,0.079,0.62
EMQ-O,"380,000",450,0.95,61.3%,61.3%,35.4%,35.4%,24.5%,94.8%,0.066,0.53
EMQ-P,"380,000",442,0.93,60.6%,60.5%,34.9%,35.0%,23.9%,94.7%,0.065,0.52
GS,"955,200",442,0.37,31.0%,31.0%,16.4%,16.4%,5.4%,88.0%,0.026,0.20
PS,"96,192",441,3.67,97.5%,97.4%,73.4%,73.4%,88.1%,98.8%,0.257,2.02
RB,"271,600",443,1.31,72.9%,72.8%,44.1%,44.2%,37.5%,96.4%,0.091,0.72
SF,"173,824",444,2.04,87.1%,87.1%,57.4%,57.4%,60.6%,97.7%,0.143,1.13


In [16]:
HELDOUT = ["active_matter", "helmholtz_staircase", "turbulent_radiative_layer_2D", "viscoelastic_instability"]
est_ft = pd.concat([estimates([n], R=4) for n in HELDOUT])
est_ft.style.format(FMT).set_caption("Held-out datasets finetuned alone: 4 GPUs, fsdp, PT9 knobs, 100 epochs")

,W,draws/rank/epoch,λ = N/W,window cov.,cov. (sim),exact-repeat draws,repeat (sim),windows drawn ≥2×,frame-overlap draws (sim),same-step shared / epoch,pair overlap / epoch
dataset,,,,,,,,,,,
ACT,"13,125",4000,121.90,100.0%,100.0%,99.2%,99.2%,100.0%,100.0%,3.657,1219.05
HS,"18,304",4000,87.41,100.0%,100.0%,98.9%,98.9%,100.0%,99.9%,2.622,874.13
TRL2,"6,840",4000,233.92,100.0%,100.0%,99.6%,99.6%,100.0%,100.0%,7.018,2339.18
VE,"5,422",4000,295.09,100.0%,100.0%,99.7%,99.7%,100.0%,100.0%,8.853,2950.94


For the single-dataset finetunes, every micro-batch comes from the one dataset, so each rank
draws `B · max_samples = 4000` windows per epoch. That is still below `W` for all four (VE has
5,422), so no recycle happens within an epoch. Within one rank-epoch there are still no exact
repeats, but across the 4 ranks of one epoch most windows are already drawn more than once.

In [17]:
# Recycling within an epoch needs B * (micro-batches of d per rank-epoch) > W_d.
for n in HELDOUT:
    W = wr.n_windows(reg[n]["splits"]["train"], T_IN, T_OUT, 1)
    print(f"{CODE[n]:5s} W={W:6,}  draws/rank/epoch={B * MS}  recycles within an epoch: {B * MS > W}"
          f"  → max_samples at which it would: {W // B + 1:,}")

ACT   W=13,125  draws/rank/epoch=4000  recycles within an epoch: False  → max_samples at which it would: 6,563
HS    W=18,304  draws/rank/epoch=4000  recycles within an epoch: False  → max_samples at which it would: 9,153
TRL2  W= 6,840  draws/rank/epoch=4000  recycles within an epoch: False  → max_samples at which it would: 3,421
VE    W= 5,422  draws/rank/epoch=4000  recycles within an epoch: False  → max_samples at which it would: 2,712
